# Bài học 18 (tiếp theo): Biên nhận Chứng minh một *Con Người* Đã Ủy Quyền Thực Hiện Hành Động

Bài học chứng minh **đại diện** đã làm gì và **cổng** đã quyết định gì. Sổ ghi chép này bổ sung nửa còn thiếu: bằng chứng rằng một **con người được định danh** đã phê duyệt **hành động chính xác** — một chữ ký riêng biệt do con người giữ, ký trên toàn bộ hành động chuẩn tắc, được xác minh offline.

Cả hai hiện vật ở đây đều sử dụng **cấu trúc phong bì giống như các biên nhận của bài học**: một tải trọng phẳng với trường `type`, được ký bằng Ed25519 trực tiếp trên các byte JCS chuẩn tắc, kèm theo đối tượng `signature` có cấu trúc (và bị loại khỏi các byte được ký). Biên nhận phê duyệt là một `type` mới (`human.approval.v1`) bên cạnh loại hành động, vì vậy một `verify_chain` bao phủ cả hai loại hiện vật với cùng đoạn mã mà bạn đã xây dựng trong sổ ghi chính. Biên nhận phê duyệt của con người đây là một thành phần giáo dục được định nghĩa ở đây, không phải loại biên nhận do draft-farley-acta-signed-receipts định nghĩa.

Một nâng cấp có chủ ý so với bộ xác minh demo trong sổ ghi chính: bộ xác minh đây giải quyết `signature.key_id` đối chiếu với **đăng ký khóa cố định** thay vì tin tưởng khóa công khai mang bên trong biên nhận. Đây là tư thế sản xuất được danh sách kiểm tra của bài học đề xuất ("công bố khóa công khai xác minh"), và điều đó làm cho việc làm giả trở thành từ chối thay vì một cách vượt qua bằng khóa tự mang.

Quy tắc sổ ghi này dạy: **phê duyệt có chữ ký không phải là quyền hạn tự nó.** Quyền hạn chỉ tồn tại nếu biên nhận phê duyệt và biên nhận hành động vẫn liên kết với cùng hành động chuẩn tắc vào thời điểm thi hành, theo phiên bản chính sách, khóa, và hạn sử dụng vẫn còn hiệu lực, và một phê duyệt chưa bị sử dụng. Mỗi thất bại đều từ chối với **lý do riêng biệt**, để bạn có thể phân biệt *quyền hạn đã hết hạn* với *hành động được thi hành đã thay đổi*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Hành động chính xác

Đơn vị phê duyệt là **đối tượng hành động chuẩn xác** — không phải một nhãn mơ hồ như "phê duyệt hoàn tiền," mà là hành động chính xác, được định nghĩa đầy đủ. Việc ký duyệt toàn bộ đối tượng (và tạo ra một bản tóm tắt từ nó) chính là cách cho phép chúng ta sau này chứng minh con người đã phê duyệt *hành động này* và không phải hành động nào khác.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Một phong bì, hai cơ quan

Mỗi biên nhận là phong bì của bài học: một tải trọng phẳng với trường `type`, cộng với một đối tượng `signature` (`alg`, `sig`, `key_id`) **không** phải là phần của các byte được ký. `verify_envelope` là kiểm tra cấu trúc + chữ ký chung cho cả hai loại biên nhận; việc nó giải quyết `signature.key_id` với **đăng ký khóa được ghim** nào chính là điều giữ cho các cơ quan được tách biệt:

- **biên nhận phê duyệt** (`human.approval.v1`) — người phê duyệt được đặt tên, hành động chuẩn chính thức đầy đủ **và tóm tắt của nó**, `policy_version`, dấu thời gian phát hành + hết hạn. Việc sử dụng một lần được theo dõi ở cấp chuỗi.
- **biên nhận hành động** (`agent.action.v1`) — danh tính đại lý, `run_id`, cùng một tóm tắt hành động chuẩn **cùng**, kết quả thực thi + dấu thời gian, và `parent_approval_ref`: `receipt_hash` của phê duyệt, quy ước giống như `previous_receipt_hash` trong chuỗi của bài học.

Trường `action_digest` chung là điểm kết nối mà ràng buộc phụ thuộc vào. `key_id` nằm trong đối tượng chữ ký chỉ như một gợi ý tra cứu: chuyển nó tới một khóa ghim khác làm kiểm tra chữ ký bị lỗi, nên nó không mang lại gì.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: nơi quyết định ràng buộc thực sự diễn ra

`verify_chain` **không phải** là một bao bọc tiện lợi cho hai lần kiểm tra chữ ký. Đây là nơi duy nhất mà `action_digest` chuẩn chia sẻ, tính **mới mẻ** của chính sách / khoá / hạn dùng phê duyệt, và việc **tiêu thụ một lần duy nhất** của phê duyệt được kiểm tra cùng nhau, đối chiếu với hành động đang được thực thi *ngay bây giờ*.

Mỗi lỗi từ chối đều có một **lý do riêng biệt**, để người đọc từ chối có thể biết được quyền hạn đã cũ (chính sách thay đổi, khoá xoay vòng, phê duyệt hết hạn, phê duyệt đã được tiêu thụ) hay hành động được thực thi đã thay đổi mà phê duyệt vẫn còn hiệu lực (thay thế digest).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Những gì binding bắt được

Mỗi trường hợp dưới đây đều thất bại **đóng** với một **lý do riêng biệt**. Khối đầu tiên là tập hợp kinh điển (can thiệp, đại diện lúng túng, phát lại, giả mạo trên bất kỳ quyền hạn nào, đầu vào bị lỗi). Khối thứ hai là cặp làm cho thuộc tính trở thành thật thay vì được khẳng định:

- **quyền hạn lỗi thời** — chữ ký vẫn hợp lệ, nhưng phiên bản chính sách đã thay đổi, khóa người duyệt đã bị xoay vòng ra khỏi registry được ghim, hoặc sự phê duyệt đã hết hạn trước khi thực thi;
- **thay thế bản tóm tắt** — một biên nhận hành động đã được ký hợp lệ mà `parent_approval_ref` trỏ đến một sự phê duyệt *thực sự*, nhưng bản tóm tắt hành động chính thức của sự phê duyệt đó không khớp với hành động thực tế đang được thực thi.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Điều này chứng minh — và điều nó không chứng minh

**Chứng minh:** một con người được đặt tên đã phê duyệt *chính xác hành động chuẩn này* (hành động đầy đủ + tóm tắt, được ký bằng một khóa được xác định từ một registry được ghim), và đại lý đã thực hiện *chính xác hành động đã được phê duyệt đó* (cùng tóm tắt, hóa đơn gắn với sự phê duyệt bằng `receipt_hash`, quy ước chuỗi riêng của bài học) — trong khi phiên bản chính sách, khóa, và thời hạn của sự phê duyệt vẫn còn hiệu lực, chính xác một lần. Nếu bất kỳ bên nào thay đổi, chuỗi sẽ đóng lại, và lý do từ chối sẽ cho bạn biết **đặc tính nào** bị phá vỡ: quyền hạn cũ so với hành động đã thay đổi.

**Không chứng minh:** rằng giao diện phê duyệt đã hiển thị cho người dùng những gì họ nghĩ họ đang ký (WYSIWYS là một vấn đề riêng), rằng khóa không bị ép buộc hoặc bị đánh cắp trước khi quay vòng, hoặc rằng các tác động tiếp theo khớp với hành động đó. Đã ký ≠ đã được ủy quyền: một chữ ký hợp lệ trên chính sách cũ, khóa đã quay vòng, cửa sổ hết hạn, hoặc một tóm tắt khác không mang lại giá trị gì ở đây.

Hai loại hóa đơn chia sẻ phong bì của bài học và một đường dẫn mã `verify_chain` duy nhất có mục đích: liên kết bạn xây dựng cho hóa đơn hành động trong sổ chính là cùng mã đó kiểm tra sự phê duyệt của con người. Một hợp đồng xác thực, quyền hạn riêng được ghim, kết nối bởi tóm tắt hành động chuẩn và không gì khác.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
